## SETUP

In [1]:
# Imports
import duckdb
import pandas as pd
import plotly.express as px

In [2]:
# Load dataset
df = pd.read_csv("pizza_sales.csv")
df.head()

,pizza_id,order_id,pizza_name_id,quantity,order_date,order_time,unit_price,total_price,pizza_size,pizza_category,pizza_ingredients,pizza_name
0,1,1,hawaiian_m,1,2015-01-01,11:38:36,13.25,13.25,M,Classic,"Sliced Ham, Pineapple, Mozzarella Cheese",The Hawaiian Pizza
1,2,2,classic_dlx_m,1,2015-01-01,11:57:40,16.00,16.00,M,Classic,"Pepperoni, Mushrooms, Red Onions, Red Peppers,...",The Classic Deluxe Pizza
2,3,2,five_cheese_l,1,2015-01-01,11:57:40,18.50,18.50,L,Veggie,"Mozzarella Cheese, Provolone Cheese, Smoked Go...",The Five Cheese Pizza
3,4,2,ital_supr_l,1,2015-01-01,11:57:40,20.75,20.75,L,Supreme,"Calabrese Salami, Capocollo, Tomatoes, Red Oni...",The Italian Supreme Pizza
4,5,2,mexicana_m,1,2015-01-01,11:57:40,16.00,16.00,M,Veggie,"Tomatoes, Red Peppers, Jalapeno Peppers, Red O...",The Mexicana Pizza


In [3]:
# Check for missing values
df.isnull().sum()

pizza_id             0
order_id             0
pizza_name_id        0
quantity             0
order_date           0
order_time           0
unit_price           0
total_price          0
pizza_size           0
pizza_category       0
pizza_ingredients    0
pizza_name           0
dtype: int64

## Convert date/time columns to datetime for correct DuckDB SQL operations (e.g., EXTRACT, strftime)

In [4]:
df['order_date'] = pd.to_datetime(df['order_date'], format='mixed', dayfirst=False)
df['order_time'] = pd.to_datetime(df['order_time'], format='%H:%M:%S')
df.sample()

,pizza_id,order_id,pizza_name_id,quantity,order_date,order_time,unit_price,total_price,pizza_size,pizza_category,pizza_ingredients,pizza_name
20358,20359,8941,hawaiian_l,1,2015-05-30,1900-01-01 16:53:40,16.5,16.5,L,Classic,"Sliced Ham, Pineapple, Mozzarella Cheese",The Hawaiian Pizza


In [5]:
# Register table
duckdb.register("pizza", df)

## 1. TOTAL REVENUE

In [6]:
query = """
SELECT ROUND(SUM(total_price), 2) AS total_revenue
FROM pizza;
"""

total_rev_df = duckdb.sql(query).df()

import plotly.graph_objects as go

fig = go.Figure(go.Indicator(
    mode="number",
    value=total_rev_df["total_revenue"][0],
    title={"text": "Total Revenue ($)"}
))

fig.show()

## 2. Top 5 Best-Selling Pizzas (Revenue)

In [7]:
query = """
SELECT pizza_name, 
       SUM(total_price) AS revenue
FROM pizza
GROUP BY pizza_name
ORDER BY revenue DESC
LIMIT 5;
"""

top_pizza_df = duckdb.sql(query).df()

fig = px.bar(
    top_pizza_df,
    x="revenue",
    y="pizza_name",
    orientation="h",
    title="Top 5 Best-Selling Pizzas by Revenue",
)

fig.show()

## 3. Sales by Month

In [8]:
query = """
SELECT 
    strftime('%Y-%m', order_date) AS month,
    SUM(total_price) AS revenue
FROM pizza
GROUP BY month
ORDER BY month;
"""

monthly_sales_df = duckdb.sql(query).df()

fig = px.line(
    monthly_sales_df,
    x="month",
    y="revenue",
    title="Monthly Revenue Sales",
    markers=True
)

fig.show()

## 4. Peak Ordering Hours

In [9]:
query = """
SELECT 
    EXTRACT(HOUR FROM order_time) AS hour,
    COUNT(order_id) AS orders
FROM pizza
GROUP BY hour
ORDER BY hour;
"""

hourly_df = duckdb.sql(query).df()

fig = px.bar(
    hourly_df,
    x="hour",
    y="orders",
    title="Orders by Hour of Day"
)

fig.show()

## 5. Average Order Value (AOV)

In [10]:
query = """
SELECT AVG(order_total) AS avg_order_value
FROM (
    SELECT order_id, SUM(total_price) AS order_total
    FROM pizza
    GROUP BY order_id
) AS orders;
"""

aov_df = duckdb.sql(query).df()

fig = go.Figure(go.Indicator(
    mode="number",
    value=aov_df["avg_order_value"][0],
    title={"text": "Average Order Value ($)"}
))

fig.show()

## 6. Sales by Category

In [11]:
query = """
SELECT pizza_category, 
       SUM(quantity) AS total_quantity, 
       SUM(total_price) AS revenue
FROM pizza
GROUP BY pizza_category
ORDER BY revenue DESC;
"""

category_df = duckdb.sql(query).df()

fig = px.bar(
    category_df,
    x="pizza_category",
    y="revenue",
    title="Revenue by Pizza Category",
)

fig.show()

fig = px.bar(
    category_df,
    x="pizza_category",
    y=["total_quantity", "revenue"],
    barmode="group",
    title="Category Performance: Quantity vs Revenue"
)

fig.show()

## 7. Quantity Sold by Size

In [12]:
query = """
SELECT pizza_size, 
       SUM(quantity) AS total_quantity
FROM pizza
GROUP BY pizza_size
ORDER BY total_quantity DESC;
"""

size_df = duckdb.sql(query).df()

fig = px.bar(
    size_df,
    x="pizza_size",
    y="total_quantity",
    title="Total Quantity Sold by Pizza Size"
)

fig.show()